# mcp_attack quickstart

One linear demo of the attack harness: `audit_then_attack` against a real stand
if it is up, otherwise a deterministic in-memory `CallableAdapter` fallback.
No Docker and no API key are required for the default path.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "mcp_attack").is_dir():
            return p
    raise RuntimeError("could not find the repo root (looked for a mcp_attack/ directory)")


REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from mcp_attack.adapters.callable_adapter import CallableAdapter
from mcp_attack.audit_plan import extract_ranked_findings, select_variants_by_audit
from mcp_attack.catalog.generator import StaticCatalogGenerator
from mcp_attack.config import TargetBinding
from mcp_attack.detectors.literal import LiteralDetector
from mcp_attack.models import Channel, ChannelRole, Principal
from mcp_attack.pipeline import audit_then_attack
from mcp_attack.reporting import emit_html
from mcp_attack.runner import run_matrix
from mcp_attack.tracer import JSONLTracer
from tests.fixtures.fake_memory_target import FakeToolPoisonableApp, FakeVulnerableMemoryApp, build_adapter

print("repo root:", REPO_ROOT)

## 1. One call: `audit_then_attack`

Tries the stand at `localhost:8600`. If that is down, falls back to an in-process
vulnerable memory so the notebook still finishes.

In [ ]:
import json
from urllib.request import urlopen

AUDIT = "examples/genai_invest_stand.audit.json"
channels = [
    Channel(role=ChannelRole.ATTACKER, principal=Principal(principal_id="1001", credential_ref="A")),
    Channel(role=ChannelRole.VICTIM, principal=Principal(principal_id="1002", credential_ref="B")),
]
target = TargetBinding(kind="openai_compat", binding={"base_url": "http://127.0.0.1:8600/v1", "model": "m"})

stand_up = False
try:
    urlopen("http://127.0.0.1:8600/v1/models", timeout=0.4)
    stand_up = True
except Exception:
    stand_up = False

fallback = build_adapter(FakeVulnerableMemoryApp(), with_ingest=True, with_tool_staging=True,
                         supported_tool_vectors=["web_search", "email"])
adapter = None if stand_up else fallback
print("stand reachable:" , stand_up, "→", "live adapter" if stand_up else "CallableAdapter fallback")

report = audit_then_attack(
    AUDIT, target, channels, pool="neutral", top_n=8,
    detector=LiteralDetector(), adapter=adapter,
)
print("overall ASR:", report.overall_asr.display)
print("counts:", dict(report.counts_by_verdict))
print("first 8 variant ids:", [r.variant_id for r in report.results])

## 2. Ranked order from the shipped audit JSON

In [ ]:
audit_doc = json.load(open(AUDIT, encoding="utf-8"))
ranked = extract_ranked_findings(audit_doc)
print(f"{len(ranked)} FAIL findings, most severe first:")
for rf in ranked[:8]:
    print(f"  {rf.severity or '?':8} {rf.rule_id:9} amg={rf.owasp_amg_category or '-':28} tech={rf.technique_category or '-'}")

all_variants = StaticCatalogGenerator(["mcp_attack/catalog/prompts/generic"]).generate()
ordered, limitations = select_variants_by_audit(all_variants, AUDIT, top_n=8)
print("top 8 after ranked:", [v.id for v in ordered])
print("limitations:", limitations)

## 3. Indirect tool vector (email / web-search) on the fallback adapter

In [ ]:
tool_pool = StaticCatalogGenerator(["mcp_attack/catalog/prompts/generic/generic_tool_websearch_injection"]).generate()
app = FakeToolPoisonableApp()
tool_adapter = build_adapter(app, with_tool_staging=True, supported_tool_vectors=["web_search"])
variant = tool_pool[0]
variant.tool_stage["tool_name"] = "duckduckgo_search"
tracer_tool = JSONLTracer()
tool_report = run_matrix([variant], [channels[1]], tool_adapter, LiteralDetector(), tracer_tool)
tracer_tool.close()
r = tool_report.results[0]
print("verdict:", r.verdict.value, "laundering:", r.laundering_detected, "channel:", r.delivery_channel)

## 4. LLM domain adaptation (skipped without a key / endpoint)

In [ ]:
try:
    from mcp_attack.mutation.domain import DomainProfile
    from mcp_attack.mutation.llm_client import LLMClient, LLMClientConfig
    from mcp_attack.mutation.techniques import DomainAdaptationTechnique
    base_url = os.environ.get("MCP_ATTACK_MUTATION_URL")
    model = os.environ.get("MCP_ATTACK_MUTATION_MODEL")
    if not base_url or not model:
        raise RuntimeError("set MCP_ATTACK_MUTATION_URL and MCP_ATTACK_MUTATION_MODEL to run this cell")
    seed = StaticCatalogGenerator(["mcp_attack/catalog/prompts/generic/generic_memory_prompt_injection"]).generate()[0]
    llm = LLMClient(LLMClientConfig(base_url=base_url, model=model))
    adapted = DomainAdaptationTechnique(DomainProfile(domain="customer support")).mutate(seed, llm=llm)
    print(adapted.inject_turns[0][:240])
except Exception as exc:
    print("skip (needs a local/remote attacker LLM):", type(exc).__name__, exc)

## 5. HTML dashboard + prompt vs response join

In [ ]:
from IPython.display import HTML, display

html = emit_html(report)
display(HTML(html))

print("--- success vs miss (first results) ---")
for row in report.results[:8]:
    print(f"{row.verdict.value:14} {row.variant_id}")